# 4_inter_intra_classification.ipynb

**Pipeline position:** Step 4 — runs *after* `key_authors_graph.ipynb` (step 3).

**Inputs (per campaign):** one `classification_sweep_indicators/{graph_filtering_setting}/`
parquet file from step 3, selected by the sweep-config parameters set below
(`topic_method`, `topic_n`, `edge_types`, `top_percentage`). Author-level, indexed by
`accountid`, with a baked-in `io_drive_label` column for the IO/non-IO ground truth.

**Experiments:**
1. **Intra-campaign** — train/test on the same campaign, varying the training fraction.
2. **Inter-campaign (LOO)** — train on the other campaigns, test on the held-out target.

**Outputs:** Results CSV, aggregated CSV, AUC plot, Macro-F1 plot.

# Imports

In [ ]:
from __future__ import annotations

# File system
from pathlib import Path
import gc
import math
import time
import traceback

# Data
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt

# sklearn & models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.model_selection import train_test_split

# Utilities
from functools import wraps


# Parameters

> Papermill will inject values into the tagged cell below.
> `CAMPAIGNS` lists the campaigns to run the sweep over. The `topic_method`, `topic_n`,
> `edge_types`, `top_percentage`, and `graph_filtering_setting` values are used to build
> each campaign's exact step-3 indicator file path — see the Configuration cell.

In [ ]:
# Campaigns — each must have a completed step-3 sweep run at the standard location
CAMPAIGNS = ["Ecuador_Test", "Cuba_Test"]

output_dir = "../results/Labeled_Datasets/step_4_inter_intra_classification"

# Sweep configuration — used to build each campaign's exact indicator file path
graph_filtering_setting = "all_authors"  # "all_authors" or "graph_authors"
topic_method = "BERTopic"
topic_n = 256
edge_types = "similarity+text_similarity+interactions"
top_percentage = 1.0

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]

MODELS = ["xgboost", "random_forest", "logistic_regression"]

# Every (campaign, fraction, model, experiment_type) combination is repeated once per
# seed below, which re-draws the stratified training sample and re-seeds the
# classifier. The 80/20 test split always uses RANDOM_STATE, so every replicate for a
# given campaign is scored against the same held-out set.
SEEDS = [42, 123, 456, 789, 1024]

RANDOM_STATE = 42
TEST_SIZE    = 0.20


# Configuration

In [ ]:
def build_indicator_path(campaign: str) -> Path:
    """Build a campaign's step-3 indicator file path from the sweep config.

    Args:
        campaign: Campaign name (matches its results directory name).

    Returns:
        Path to the expected `classification_sweep_indicators` parquet file.
    """
    base_dir = Path(
        f"../results/Labeled_Datasets/{campaign}/{campaign}_part_all"
        f"/step_3_key_authors_graph/topic_col_{topic_method}_topic_{topic_n}"
        f"_top_percentage_1"
    )
    sweep_filename = (
        f"accounts_graph_{topic_method}_topic_{topic_n}_{edge_types}"
        f"_top_percentage_{top_percentage}_per_topic"
        f"_topic_col_{topic_method}_topic_{topic_n}_undirected_indicators.parquet"
    )
    return base_dir / "classification_sweep_indicators" / graph_filtering_setting / sweep_filename


campaign_base_dirs = {campaign: build_indicator_path(campaign) for campaign in CAMPAIGNS}

for campaign, path in campaign_base_dirs.items():
    if not path.exists():
        raise FileNotFoundError(
            f"No data found for '{campaign}' at: {path}\n"
            f"Has the pipeline completed step 3 for this campaign?"
        )


# Logs

In [ ]:
output_path = Path(output_dir).expanduser()
output_path.mkdir(parents=True, exist_ok=True)
REPORT_PATH = output_path / "progress_report.txt"


def write_report(msg: str) -> None:
    """Write a timestamped message to stdout and the progress report.

    Args:
        msg: Message to write.
    """
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} | {msg}"
    print(line)
    with open(REPORT_PATH, "a", encoding="utf-8") as report_file:
        report_file.write(line + "\n")


write_report(f"Campaign paths: {campaign_base_dirs}")
write_report(f"Output path: {output_path}")


def report_done(func):
    """Log completion or failure for a function.

    Args:
        func: Function to wrap.

    Returns:
        Wrapped function.
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as exc:
            write_report(f"FAILED: {func.__name__} | {exc}")
            raise
    return wrapper

# Classification

In [ ]:
def get_classifier(model_name: str, random_state: int = RANDOM_STATE, **kwargs):
    """Return a sklearn-compatible classifier by name.

    Args:
        model_name: One of "xgboost"/"xgb", "random_forest"/"rf",
            "logistic_regression"/"lr".
        random_state: Random seed forwarded to the classifier constructor.
        **kwargs: Additional keyword arguments passed to the constructor.

    Returns:
        An untrained sklearn-compatible classifier instance.

    Raises:
        ValueError: If model_name is not recognised.
    """
    name = model_name.lower()
    if name in ("xgboost", "xgb"):
        return XGBClassifier(random_state=random_state, eval_metric="logloss", **kwargs)
    if name in ("random_forest", "rf"):
        return RandomForestClassifier(random_state=random_state, **kwargs)
    if name in ("logistic_regression", "lr"):
        return LogisticRegression(random_state=random_state, solver="liblinear", max_iter=1000, **kwargs)
    raise ValueError(
        f"Unsupported model_name: '{model_name}'. "
        "Supported: xgboost/xgb, random_forest/rf, logistic_regression/lr."
    )

# Data Loading

Each campaign in `campaign_base_dirs` points to one step-3 `classification_sweep_indicators`
parquet file — an author-level feature DataFrame with a baked-in `io_drive_label` column.

`load_country_data` loads one campaign's file. `load_multiple_campaigns` loads and
concatenates several campaigns (used for the LOO inter-campaign training pool).

In [ ]:
def load_country_data(
    campaign_name: str,
    indicator_path: str | Path,
    prefix_ids: bool = False,
) -> tuple[pd.DataFrame, set]:
    """Load the node-indicator DataFrame and IO author set for one campaign.

    Args:
        campaign_name: Campaign label, used for logging and (if `prefix_ids`) as the
            `accountid` prefix.
        indicator_path: Path to the campaign's `classification_sweep_indicators`
            parquet file.
        prefix_ids: If True, prepend `"{campaign_name}_"` to every `accountid` so
            authors from different campaigns don't collide when concatenated.

    Returns:
        df_features: Author-level DataFrame indexed by `accountid`, numeric columns
            only, ready for classification.
        io_set: Set of `accountid` strings labelled as IO.

    Raises:
        FileNotFoundError: If the indicator file doesn't exist.
        ValueError: If the file is missing the `io_drive_label` column.
    """
    campaign_name = campaign_name.strip()
    path = Path(indicator_path).expanduser()

    if not path.exists():
        raise FileNotFoundError(f"No indicator file for '{campaign_name}' at: {path}")

    df_features = pd.read_parquet(path)
    if "io_drive_label" not in df_features.columns:
        raise ValueError(f"'{path}' is missing the expected 'io_drive_label' column.")

    io_set = set(df_features.index[df_features["io_drive_label"] == 1].astype(str))
    df_features = df_features.drop(columns=["io_drive_label"])

    if prefix_ids:
        df_features.index = campaign_name + "_" + df_features.index.astype(str)
        io_set = {f"{campaign_name}_{aid}" for aid in io_set}

    n_io_in_graph = len(set(df_features.index.astype(str)) & io_set)
    write_report(
        f"Loaded '{campaign_name}' ({path}) | {len(df_features)} authors in graph "
        f"| {len(io_set)} IO authors (dataset) | {n_io_in_graph} IO in graph"
    )
    return df_features, io_set


def load_multiple_campaigns(campaign_paths: dict[str, Path]) -> tuple[pd.DataFrame, set]:
    """Load and concatenate node-indicator data for several campaigns.

    Args:
        campaign_paths: Mapping from campaign name to its indicator file path.

    Returns:
        Combined author-level DataFrame (index prefixed with campaign name) and the
        union of all IO author sets.

    Raises:
        FileNotFoundError: If any campaign's file can't be loaded.
    """
    frames: list[pd.DataFrame] = []
    combined_io: set = set()

    for campaign_name, path in campaign_paths.items():
        df, io_set = load_country_data(campaign_name, path, prefix_ids=True)
        frames.append(df)
        combined_io.update(io_set)

    return pd.concat(frames, axis=0), combined_io


In [ ]:
# Probe-load one campaign to see the real feature-column set used for classification.
_df_probe, _ = load_country_data(CAMPAIGNS[0], campaign_base_dirs[CAMPAIGNS[0]])
FEATURE_COLUMNS_USED: list[str] = _df_probe.select_dtypes(include="number").columns.tolist()
write_report(
    f"Feature columns used ({len(FEATURE_COLUMNS_USED)}, "
    f"probed from '{CAMPAIGNS[0]}'): {FEATURE_COLUMNS_USED}"
)
del _df_probe
gc.collect()


# Classification Engine

Mirrors `io_classification` from `key_authors_graph.ipynb` exactly, with an
additional cross-campaign test path so that the same function handles both
intra-campaign (internal train/test split) and inter-campaign (external test
set) experiments.

In [ ]:
@report_done
def io_classification(
    node_indicators_df: pd.DataFrame,
    io_authors_set: set,
    model_name: str,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
    test_node_indicators_df: pd.DataFrame | None = None,
    test_io_authors_set: set | None = None,
) -> tuple:
    """Train and evaluate an IO classifier.

    Classification logic mirrors ``key_authors_graph.ipynb`` exactly.

    Two modes:

    * **Intra-campaign** (default) — performs ``train_test_split`` on
      *node_indicators_df*.
    * **Cross-campaign** — bypasses the internal split and uses the explicitly
      supplied *test_node_indicators_df* / *test_io_authors_set*. Feature
      columns must match exactly between train and test DataFrames.

    Args:
        node_indicators_df: Training-set author DataFrame, indexed by
            ``accountid``, numeric columns are used as features.
        io_authors_set: Set of IO ``accountid`` strings for the training data.
        model_name: Model identifier — see :func:`get_classifier`.
        test_size: Fraction held out for testing in intra-campaign mode.
        random_state: Random seed used for splitting and the classifier.
        test_node_indicators_df: External test DataFrame for cross-campaign
            mode.  Must have the same numeric columns as the training frame.
        test_io_authors_set: IO author set matching *test_node_indicators_df*.

    Returns:
        tuple(clf, report, auc, macro_f1):
            clf       – Trained classifier instance.
            report    – Classification report string.
            auc       – ROC-AUC score (0.5 when only one class is present).
            macro_f1  – Macro-averaged F1 score.

    Raises:
        ValueError: If the training set is empty or train/test feature columns
            do not match in cross-campaign mode.
    """
    if node_indicators_df.empty:
        raise ValueError("Training feature set is empty — aborting.")

    # Only numeric columns are used as model features
    train_numeric = node_indicators_df.select_dtypes(include="number")
    train_cols    = train_numeric.columns.tolist()
    X_train       = train_numeric.to_numpy()

    io_str  = set(map(str, io_authors_set))
    y_train = node_indicators_df.index.astype(str).isin(io_str).astype(int)

    # ── Cross-campaign mode ──────────────────────────────────────────────────
    if test_node_indicators_df is not None and test_io_authors_set is not None:
        write_report("Cross-campaign mode: using external test set.")

        test_numeric = test_node_indicators_df.select_dtypes(include="number")
        test_cols    = test_numeric.columns.tolist()

        # Strict alignment: fail fast if feature sets diverge
        if train_cols != test_cols:
            mismatch = set(train_cols) ^ set(test_cols)
            raise ValueError(
                f"FATAL ALIGNMENT ERROR: Train/test feature columns do not match.\n"
                f"Train cols: {len(train_cols)}  |  Test cols: {len(test_cols)}\n"
                f"Mismatched: {mismatch}"
            )

        X_test       = test_numeric.to_numpy()
        test_io_str  = set(map(str, test_io_authors_set))
        y_test       = test_node_indicators_df.index.astype(str).isin(test_io_str).astype(int)

    # ── Intra-campaign mode ──────────────────────────────────────────────────
    else:
        stratify = y_train if len(np.unique(y_train)) > 1 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X_train, y_train,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )

    write_report(
        f"  [{model_name}] Train: {len(X_train)} | Test: {len(X_test)} "
        f"| IO in test: {int(y_test.sum())}"
    )

    clf = get_classifier(model_name=model_name, random_state=random_state)
    clf.fit(X_train, y_train)

    y_pred  = clf.predict(X_test)
    y_proba = (
        clf.predict_proba(X_test)[:, 1]
        if hasattr(clf, "predict_proba")
        else y_pred.astype(float)
    )

    report   = classification_report(
        y_test, y_pred, target_names=["Non-IO", "IO"], zero_division=0
    )
    auc      = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    return clf, report, auc, macro_f1

# Experiment Sweep

For every target campaign the sweep:
1. Loads its node-indicator DataFrame.
2. Creates a **fixed 80 / 20 train/test split** (same seed → identical test set
   across all fractions, models, and seed replicates).
3. Verifies **zero overlap** between the sampled train set and the test set.
4. Runs **intra-campaign** experiments (train on the current campaign).
5. Runs **inter-campaign / LOO** experiments (train on the other campaigns pooled
   together, test on the same held-out 20 %).
6. Repeats every (fraction, model, experiment_type) combination once per seed
   in `SEEDS` (the test set stays fixed across seeds — only the training
   sample draw and the classifier are re-seeded).

In [ ]:
def _stratified_sample(
    df: pd.DataFrame,
    io_str_set: set,
    n: int,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Sample n rows from df, guaranteeing at least 1 IO and 1 non-IO author.

    Args:
        df: Author-level DataFrame to sample from.
        io_str_set: Set of IO accountid strings (already prefixed if needed).
        n: Desired sample size (≥ 2).
        random_state: Random seed.

    Returns:
        Sampled DataFrame with at most *n* rows, both classes represented.
    """
    y         = df.index.astype(str).isin(io_str_set).astype(int)
    io_idx    = df.index[y == 1].tolist()
    nonIO_idx = df.index[y == 0].tolist()

    # If only one class is available, fall back to a plain sample
    if not io_idx or not nonIO_idx:
        return df.sample(n=min(n, len(df)), random_state=random_state)

    # Proportional allocation, minimum 1 of each class
    prop_io = len(io_idx) / len(df)
    n_io    = max(1, min(int(round(n * prop_io)), len(io_idx), n - 1))
    n_nonIO = max(1, min(n - n_io, len(nonIO_idx)))

    rng           = np.random.default_rng(random_state)
    sampled_io    = rng.choice(io_idx,    size=n_io,    replace=False).tolist()
    sampled_nonIO = rng.choice(nonIO_idx, size=n_nonIO, replace=False).tolist()

    return df.loc[sampled_io + sampled_nonIO]


def run_experiment_sweep(
    campaigns: list[str],
    fractions: list[float],
    models: list[str],
    seeds: list[int],
    max_model_failures: int = 5,
) -> pd.DataFrame:
    """Run the full intra / LOO inter-campaign classification sweep, replicated
    across multiple seeds.

    For every target campaign the function:

    1. Loads the node-indicator DataFrame and IO set from `campaign_base_dirs`.
    2. Creates one **fixed 80 / 20 train/test split**, seeded by `RANDOM_STATE` —
       the held-out test set is identical across every fraction, model, and seed
       replicate for a given campaign.
    3. For each fraction and each seed, draws a **stratified sample** from the
       80 % train pool so both classes are always represented, even at 1 %.
    4. Verifies zero overlap between the sampled train set and the test set.
    5. Runs **intra-campaign** and **LOO inter-campaign** experiments for every
       model. A single model/fraction/seed failure is logged with its traceback
       and doesn't stop the sweep — the sweep aborts once `max_model_failures`
       such failures accumulate.

    Args:
        campaigns: Campaign names — each must be a key in `campaign_base_dirs`.
        fractions: Training fractions to sweep, values in (0, 1].
        models: Model name strings passed to :func:`get_classifier`.
        seeds: Random seeds to replicate every (campaign, fraction, model,
            experiment_type) combination over.
        max_model_failures: Number of per-model/fraction/seed failures to
            tolerate before re-raising.

    Returns:
        DataFrame with columns:
        `training_campaign`, `target_campaign`, `graph_filtering_setting`,
        `train_fraction`, `model`, `experiment_type`, `seed`, `auc`, `macro_f1`.

    Raises:
        FileNotFoundError: If a target or LOO campaign's data can't be loaded.
    """
    results: list[dict] = []
    n_model_failures = 0

    for target_camp in campaigns:
        write_report("=" * 60)
        write_report(f"Target campaign: {target_camp}")

        df_target, io_target = load_country_data(target_camp, campaign_base_dirs[target_camp])

        io_target_str = set(map(str, io_target))
        y_all = df_target.index.astype(str).isin(io_target_str).astype(int)

        if y_all.sum() == 0:
            write_report(
                f"SKIP {target_camp}: no IO authors in the graph — "
                "cannot produce a meaningful evaluation."
            )
            continue

        # Fixed 80/20 split, seeded by RANDOM_STATE — identical test set for
        # every fraction/model/seed replicate.
        stratify = y_all if len(np.unique(y_all)) > 1 else None
        df_train_full, df_test, y_train_full, y_test_ref = train_test_split(
            df_target, y_all,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
            stratify=stratify,
        )
        test_io_set = set(df_test.index[y_test_ref == 1].astype(str))

        write_report(
            f"  Train pool: {len(df_train_full)} | Test: {len(df_test)} "
            f"| IO in test: {int(y_test_ref.sum())}"
        )

        # ── LOO: load and pool the other campaigns ──────────────────────
        loo_camps = [c for c in campaigns if c != target_camp]
        df_inter_all, io_inter_all = load_multiple_campaigns(
            {c: campaign_base_dirs[c] for c in loo_camps}
        )
        write_report(
            f"  LOO train pool: {len(df_inter_all)} authors from {loo_camps}"
        )

        # ── Fraction × seed × model sweep ─────────────────────────────
        for frac in fractions:
            n_intra = max(2, min(int(round(len(df_train_full) * frac)), len(df_train_full)))
            n_inter = max(2, min(int(round(len(df_inter_all) * frac)), len(df_inter_all)))

            for seed in seeds:
                # ── Intra: stratified sample from 80 % train pool, this seed ──
                if frac >= 1.0:
                    df_train_frac = df_train_full
                else:
                    df_train_frac = _stratified_sample(
                        df_train_full, io_target_str, n_intra, seed
                    )

                overlap = set(df_train_frac.index.astype(str)) & set(df_test.index.astype(str))
                assert len(overlap) == 0, (
                    f"FATAL: {len(overlap)} authors appear in both train and test sets "
                    f"for '{target_camp}' at fraction {frac}, seed {seed}!"
                )

                train_io_frac = set(df_train_frac.index.astype(str)) & io_target_str
                write_report(
                    f"  frac={frac*100:.1f}% seed={seed} | train_n={len(df_train_frac)} "
                    f"| IO_in_train={len(train_io_frac)}"
                )

                # ── LOO training pool for this seed (resampled if frac < 1) ──
                if frac >= 1.0:
                    df_inter_frac = df_inter_all
                else:
                    df_inter_frac = _stratified_sample(
                        df_inter_all, set(map(str, io_inter_all)), n_inter, seed
                    )
                inter_io_frac = (
                    set(df_inter_frac.index.astype(str))
                    & set(map(str, io_inter_all))
                )

                for model_name in models:
                    # ─ Intra ─────────────────────────────────────────
                    try:
                        _, _, auc, macro_f1 = io_classification(
                            node_indicators_df=df_train_frac,
                            io_authors_set=train_io_frac,
                            model_name=model_name,
                            random_state=seed,
                            test_node_indicators_df=df_test,
                            test_io_authors_set=test_io_set,
                        )
                        results.append({
                            "training_campaign":       target_camp,
                            "target_campaign":         target_camp,
                            "graph_filtering_setting": graph_filtering_setting,
                            "train_fraction":          frac,
                            "model":                   model_name,
                            "experiment_type":         "intra",
                            "seed":                    seed,
                            "auc":                     auc,
                            "macro_f1":                macro_f1,
                        })
                    except Exception:
                        n_model_failures += 1
                        write_report(
                            f"  ERROR intra | {target_camp} | {model_name} | "
                            f"frac={frac} | seed={seed} "
                            f"(failure {n_model_failures}/{max_model_failures})\n"
                            f"{traceback.format_exc()}"
                        )
                        if n_model_failures >= max_model_failures:
                            raise

                    # ─ Inter (LOO) ────────────────────────────────
                    try:
                        _, _, auc, macro_f1 = io_classification(
                            node_indicators_df=df_inter_frac,
                            io_authors_set=inter_io_frac,
                            model_name=model_name,
                            random_state=seed,
                            test_node_indicators_df=df_test,
                            test_io_authors_set=test_io_set,
                        )
                        results.append({
                            "training_campaign":       ",".join(loo_camps),
                            "target_campaign":         target_camp,
                            "graph_filtering_setting": graph_filtering_setting,
                            "train_fraction":          frac,
                            "model":                   model_name,
                            "experiment_type":         "inter",
                            "seed":                    seed,
                            "auc":                     auc,
                            "macro_f1":                macro_f1,
                        })
                    except Exception:
                        n_model_failures += 1
                        write_report(
                            f"  ERROR inter | {target_camp} | {model_name} | "
                            f"frac={frac} | seed={seed} "
                            f"(failure {n_model_failures}/{max_model_failures})\n"
                            f"{traceback.format_exc()}"
                        )
                        if n_model_failures >= max_model_failures:
                            raise

        # Free memory before next campaign
        del df_target, df_train_full, df_inter_all
        gc.collect()

    return pd.DataFrame(results)


# Execution

In [ ]:
write_report("Starting cross-campaign IO detection sweep.")
write_report(f"Campaigns  : {CAMPAIGNS}")
write_report(f"Fractions  : {FRACTIONS}")
write_report(f"Models     : {MODELS}")
write_report(f"Seeds      : {SEEDS}")
write_report(f"Graph filter: {graph_filtering_setting}")
write_report(f"Output dir : {output_path}")

results_df = run_experiment_sweep(
    campaigns=CAMPAIGNS,
    fractions=FRACTIONS,
    models=MODELS,
    seeds=SEEDS,
)

if results_df.empty:
    write_report(
        "WARNING: No results collected. "
        "Ensure all campaigns have completed step-3 sweep runs."
    )
else:
    write_report(f"Sweep complete — {len(results_df)} result rows collected.")
    display(results_df.head(12))


# Save Results

In [ ]:
if not results_df.empty:
    csv_path = output_path / f"experiment_results_{graph_filtering_setting}.csv"
    results_df.to_csv(csv_path, index=False)
    write_report(f"Results saved → {csv_path}")
    display(results_df.describe())

    # ── Seed-aggregated summary (mean/std across the SEEDS replicates) ────────
    agg_path = output_path / f"experiment_results_{graph_filtering_setting}_aggregated.csv"
    results_agg = (
        results_df
        .groupby(["target_campaign", "experiment_type", "model", "train_fraction"])
        .agg(
            auc_mean=("auc", "mean"),
            auc_std=("auc", "std"),
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            n_seeds=("seed", "nunique"),
        )
        .reset_index()
    )
    results_agg.to_csv(agg_path, index=False)
    write_report(f"Seed-aggregated results saved → {agg_path}")
    display(results_agg.head(12))

# Plots

Two figures are produced — one for **AUC** and one for **Macro-F1**:

* **Dynamically sized grid** (`ceil(sqrt(n)) x ceil(n / ceil(sqrt(n)))`), one subplot per country.
* **Orange/gold** line — intra-country; **blue** line — inter-country LOO.
* **Error bars** = std across the three models × five seeds at each fraction, clipped so the bar never crosses below 0 or above 1.
* **Y-axis** fixed to `[0, 1.05]` on every subplot (both metrics are bounded in `[0, 1]`).
* **Log-scale X-axis**: 1 % → 100 %.

In [ ]:
LINES = [
    ("intra", "Intra-country", "#D4A017", "o"),         # gold/orange, circle
    ("inter", "Inter-country (LOO)", "#4472C4", "s"),   # blue, square
]

CAMPAIGN_ORDER = list(campaign_base_dirs.keys())


def plot_cross_campaign_performance(
    df_results: pd.DataFrame,
    metric: str,
    output_dir: Path | None = None,
) -> None:
    """Plot intra- vs inter-country performance.

    Produces a dynamically sized subplot grid (one subplot per country,
    sized ``ceil(sqrt(n)) x ceil(n / ceil(sqrt(n)))`` for ``n`` countries in
    ``CAMPAIGN_ORDER``).  Each subplot shows two lines — intra-country
    (orange) and inter-country LOO (blue) — with ±1 std error bars (across
    models × seeds) clipped to ``[0, 1]``, on a fixed ``[0, 1.05]`` Y-axis.

    Args:
        df_results: Output of the experiment sweep.  Must contain columns
            ``target_campaign``, ``experiment_type``, ``train_fraction``,
            and the requested *metric* column.
        metric: Column to visualise — ``"auc"`` or ``"macro_f1"``.
        output_dir: Directory to save the PNG at 300 dpi.  Not saved if
            ``None``.
    """
    campaigns = [c for c in CAMPAIGN_ORDER if c in df_results["target_campaign"].unique()]
    if not campaigns:
        write_report(f"No data to plot for metric '{metric}'.")
        return

    ncols = math.ceil(math.sqrt(len(campaigns)))
    nrows = math.ceil(len(campaigns) / ncols)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6 * ncols, 5 * nrows))
    axes_flat = np.atleast_1d(axes).flatten()

    for idx, camp in enumerate(campaigns):
        ax = axes_flat[idx]
        camp_data = df_results[df_results["target_campaign"] == camp]

        # ── Main lines (one per experiment type) ──────────────────────────────
        for exp_type, label, color, marker in LINES:
            subset = camp_data[camp_data["experiment_type"] == exp_type]
            if subset.empty:
                continue
            grouped = (
                subset.groupby("train_fraction")[metric]
                .agg(["mean", "std"])
                .reset_index()
            )
            mean = grouped["mean"]
            std  = grouped["std"].fillna(0)
            # Clip the error bar to [0, 1] since both AUC and macro-F1 are bounded there
            lower = np.maximum(mean - std, 0.0)
            upper = np.minimum(mean + std, 1.0)
            yerr  = np.vstack([mean - lower, upper - mean])
            ax.errorbar(
                grouped["train_fraction"] * 100,
                mean,
                yerr=yerr,
                fmt=f"-{marker}",
                color=color,
                label=label,
                capsize=3,
                linewidth=1.5,
                markersize=6,
                elinewidth=1.0,
            )

        # ── X-axis: log scale, fixed readable ticks ───────────────────────────
        ax.set_xscale("log")
        ax.set_xticks([1, 5, 10, 25, 50, 100])
        ax.set_xticklabels(["1", "5", "10", "25", "50", "100"])
        ax.set_xlim(0.9, 110)

        # ── Per-subplot labels and legend ─────────────────────────────────────
        ax.set_title(camp, fontweight="bold", fontsize=11)
        ax.set_xlabel("Percentage of Training Data (%)", fontsize=9)
        ax.set_ylabel(metric.upper(), fontsize=9)
        ax.legend(fontsize=8, loc="lower right")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.set_ylim(0, 1.05)

    # ── Hide unused subplot slots ─────────────────────────────────────────────
    for idx in range(len(campaigns), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    # ── Figure-level title, layout, and save ──────────────────────────────────
    fig.suptitle(
        f"Intra- vs Inter-Campaign IO Detection — {metric.upper()}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    if output_dir:
        out_path = output_dir / f"strategy_comparison_{metric}.png"
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        write_report(f"Saved plot → {out_path}")
    plt.show()

In [ ]:
if not results_df.empty:
    for metric in ["auc", "macro_f1"]:
        plot_cross_campaign_performance(
            df_results=results_df,
            metric=metric,
            output_dir=output_path,
        )

write_report("Pipeline completed.")